# IBL SPN clustering

The recordings are obtained programmatically from the public IBL Brain-Wide Map release through ONE. 
Users do not manually add the raw IBL recordings to this repository.

- Brain-Wide Map publication: https://www.nature.com/articles/s41586-025-09235-0
- Official data-release instructions: https://docs.internationalbrainlab.org/notebooks_external/2025_data_release_brainwidemap.html
- ONE documentation: https://int-brain-lab.github.io/ONE/notebooks/one_quickstart.html

An internet connection is required for the first run. Downloaded files are reused from `~/ibl_bwm_cache`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

import warnings
from one.alf.exceptions import ALFWarning
warnings.filterwarnings("ignore", category=ALFWarning, message="Multiple revisions.*")

In [2]:
import numpy as np
import pandas as pd
from one.api import ONE

from spn_figures.config import DERIVED, SOURCE
from spn_figures.io import read_csv, write_csv
from spn_figures.datasets import (
    load_ibl_insertion_exact,
    resolve_ibl_candidates_exact,
)
from spn_figures.clustering import (
    cluster_ibl_session_exact,
    ibl_plot_tables_exact,
    ibl_population_activity_table_exact,
)

CACHE_DIR = Path.home() / "ibl_bwm_cache"
ONE.setup(
    base_url="https://openalyx.internationalbrainlab.org",
    cache_dir=CACHE_DIR,
    silent=True,
)
one = ONE(password="international")
try:
    one.load_cache(tag="Brainwidemap")
except Exception:
    pass

source_dir = SOURCE / "ibl"
out_dir = DERIVED / "ibl"
out_dir.mkdir(parents=True, exist_ok=True)

manifest = read_csv(source_dir / "insertion_manifest.csv")

print(f"Loaded {len(manifest)} manuscript sessions.")

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"
Downloading: /Users/zhuojunyu/ibl_bwm_cache/Brainwidemap/tmpsb4ps002/cache.zip Bytes: 4353173


100%|██████████| 4.1515092849731445/4.1515092849731445 [00:00<00:00, 15.45it/s]


Loaded 7 manuscript sessions.


In [3]:
all_pids = one.search_insertions(
    atlas_acronym="CP",
    datasets="spikes.times.npy",
    project="brainwide",
)
metadata_cache = {}

profiles = []
correlations = []
activities = []
labels = []
selected_trials = []
resolved_insertions = []
summaries = []

for row in manifest.itertuples(index=False):
    candidates = resolve_ibl_candidates_exact(
        one,
        lab=row.lab,
        subject=row.subject,
        date=row.date,
        max_insertions_to_process=int(row.max_insertions_to_process),
        random_seeds=row.random_seeds,
        all_pids=all_pids,
        metadata_cache=metadata_cache,
    )
    if candidates.empty:
        raise RuntimeError(f"No candidate insertion found for {row.session}.")

    result = None
    payload = None
    chosen = None
    errors = []
    for candidate in candidates.itertuples(index=False):
        try:
            payload = load_ibl_insertion_exact(
                one,
                pid=candidate.pid,
                session_name=row.session,
                random_seed=int(candidate.resolution_seed),
                rt_window_s=(0.080, float(row.rt_max_s)),
                premove_gap_s=float(row.premove_gap_s),
            )
            result = cluster_ibl_session_exact(payload)
            chosen = candidate
            break
        except Exception as error:
            errors.append(f"{candidate.pid}: {error}")

    if result is None:
        raise RuntimeError(
            f"No candidate completed clustering for {row.session}.\n"
            + "\n".join(errors)
        )

    profile_table, correlation_table, plot_metadata = ibl_plot_tables_exact(result, plot_matching_seed=50)
    profiles.append(profile_table)
    correlations.append(correlation_table)
    activities.append(ibl_population_activity_table_exact(result, one=one))
    labels.append(
        pd.DataFrame(
            {
                "session": result["session"],
                "eid": result["eid"],
                "pid": result["pid"],
                "probe": result["probe"],
                "session_path": result["session_path"],
                "unit_id": result["unit_ids"],
                "lr_pref": result["lr_pref"],
                "final_label": result["final_labels"],
                "final_name": result["final_names"],
            }
        )
    )

    selected = np.sort(np.concatenate([payload["idxL"], payload["idxR"]]))
    selected_trials.append(
        pd.DataFrame(
            {
                "session": result["session"],
                "trial_id": payload["trial_ids"][selected],
                "choice_left": (payload["choice"][selected] == -1).astype(int),
                "decision_time_ms": payload["decision_time_ms"][selected],
                "signed_contrast": payload["signed_contrast"][selected],
                "random_seed": int(payload["random_seed"]),
            }
        )
    )
    resolved_insertions.append(
        {
            "session": result["session"],
            "eid": result["eid"],
            "pid": result["pid"],
            "probe": result["probe"],
            "resolution_seed": int(chosen.resolution_seed),
            "scan_rank": int(chosen.scan_rank),
            "max_insertions_to_process": int(chosen.max_insertions_to_process),
        }
    )

    counts = pd.Series(result["final_names"]).value_counts()
    summaries.append(
        {
            "session": result["session"],
            "random_seed": int(payload["random_seed"]),
            "premove_gap_s": float(payload["premove_gap_s"]),
            "rt_max_s": float(row.rt_max_s),
            "n_eligible_trials": int(len(payload["trial_ids"])),
            "n_plot_trials": int(plot_metadata["n_plot_trials"]),
            "n_units_after_original_ramp_filter": int(len(result["unit_ids"])),
            "n_units_assigned": int(np.sum(result["final_labels"] >= 0)),
            "n_dSPN_left": int(counts.get("dSPN_left", 0)),
            "n_iSPN_left": int(counts.get("iSPN_left", 0)),
            "n_dSPN_right": int(counts.get("dSPN_right", 0)),
            "n_iSPN_right": int(counts.get("iSPN_right", 0)),
        }
    )

profiles = pd.concat(profiles, ignore_index=True)
correlations = pd.concat(correlations, ignore_index=True)
activities = pd.concat(activities, ignore_index=True)
labels = pd.concat(labels, ignore_index=True)
selected_trials = pd.concat(selected_trials, ignore_index=True)
resolved_insertions = pd.DataFrame(resolved_insertions)
summary = pd.DataFrame(summaries)

profiles.to_csv(out_dir / "unit_profiles.csv.gz", index=False)
write_csv(correlations, out_dir / "correlations.csv")
activities.to_csv(out_dir / "population_activity_bins.csv.gz", index=False)
write_csv(labels, out_dir / "unit_labels.csv")
write_csv(selected_trials, out_dir / "selected_trials.csv")
write_csv(resolved_insertions, out_dir / "resolved_insertions.csv")
write_csv(summary, out_dir / "session_summary.csv")

summary

,session,random_seed,premove_gap_s,rt_max_s,n_eligible_trials,n_plot_trials,n_units_after_original_ramp_filter,n_units_assigned,n_dSPN_left,n_iSPN_left,n_dSPN_right,n_iSPN_right
0,churchlandlab | CSHL049 | 2020-01-08,70,0.01,4.0,598,182,30,24,9,8,4,3
1,zadorlab | CSH_ZAD_026 | 2020-08-17,0,0.01,4.0,1011,256,29,29,13,3,3,10
2,danlab | DY_014 | 2020-07-18,4,0.01,4.0,457,158,33,29,4,7,7,11
3,hoferlab | SWC_061 | 2020-11-23,30,0.01,4.0,576,232,26,24,9,8,3,4
4,mrsicflogellab | SWC_038 | 2020-08-01,2,0.01,4.0,604,234,38,35,3,3,26,3
5,mrsicflogellab | SWC_038 | 2020-07-31,50,0.01,4.0,461,260,21,16,3,5,3,5
6,cortexlab | KS086 | 2022-03-16,60,0.01,4.0,469,134,23,23,5,4,3,11
